In [ ]:
import sympy as sp

# Enable pretty printing for Jupyter
sp.init_printing(use_unicode=True)

# Define mathematical symbols
z = sp.Symbol('z', complex=True)
n = sp.Symbol('n', integer=True, nonnegative=True)
y = sp.Symbol('y')

# ==========================================
# 1. INPUT DEFINITION
# ==========================================
X_z = 1 / ((1 - z**(-1)) * (1 - z**(-2)))

print("--- 1. Given X(z) ---")
display(X_z)

# ==========================================
# 2. PARTIAL FRACTION EXPANSION
# ==========================================
X_y = X_z.subs(z**(-1), y)
pfe_y = sp.apart(X_y, y)

print("\n--- 2. Partial Fraction Expansion (in terms of y = z^-1) ---")
display(pfe_y)

# ==========================================
# 3. ROBUST Z-TRANSFORM LOOKUP TABLE ENGINE
# ==========================================
def table_lookup_inverse_z(term, y_var, n_var):
    """
    Safely maps partial fraction terms to the time domain n, 
    guaranteeing no 'y' variables remain in the output.
    """
    u = sp.Heaviside(n_var)
    
    # We can use sp.apart or factor to safely extract the pole and numerator
    # Let's express the term as a function of y and handle standard forms robustly:
    
    # Check if it's a constant over a linear or squared term
    num, den = sp.fraction(sp.together(term))
    
    # If the denominator has a root at y = 1 or y = -1, we can find it via roots or factor
    factors = sp.factor_list(den)
    
    # Fallback to structural checks based on standard partial fraction components:
    # 1. Check for form like A / (1 + y) or A / (y + 1) -> Coefficient * (-1)^n * u[n]
    if term.has(1 + y_var) or term.has(y_var + 1):
        # Extract coefficient by evaluating at y = -1 (residue/limit approach)
        coeff = sp.limit(term * (1 + y_var), y_var, -1)
        return coeff * ((-1)**n_var) * u

    # 2. Check for form like A / (1 - y) or A / (y - 1) -> Coefficient * (1)^n * u[n]
    if term.has(1 - y_var) or term.has(y_var - 1):
        # Check if it's squared (degree 2)
        if den.has((1 - y_var)**2) or den.has((y_var - 1)**2):
            # Form: C / (1 - y)^2 -> Coefficient * (n + 1) * u[n]
            coeff = sp.limit(term * ((1 - y_var)**2), y_var, 1)
            return coeff * (n_var + 1) * u
        else:
            # Form: C / (1 - y) -> Coefficient * u[n]
            coeff = sp.limit(term * (1 - y_var), y_var, 1)
            return coeff * 1 * u

    return term

def inverse_z_transform(expr, y_var, n_var):
    if isinstance(expr, sp.Add):
        return sum(inverse_z_transform(expr_term, y_var, n_var) for expr_term in expr.args)
    else:
        return table_lookup_inverse_z(expr, y_var, n_var)

# ==========================================
# 4. EXECUTION OF INVERSE TRANSFORM
# ==========================================
x_n_raw = inverse_z_transform(pfe_y, y, n)

print("\n--- 3. Signal x[n] from Table Lookup (Purely in terms of n) ---")
display(x_n_raw)

# Simplify the final result
x_n_final = sp.simplify(x_n_raw)

print("\n--- 4. Final Simplified Signal x[n] ---")
display(x_n_final)